In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()
PROJECT_ROOT

WindowsPath('C:/Users/huyy/AirPollutionPrediction-CNN-BiLSTM')

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
STATION_SPLIT_DIR = DATA_DIR / "station_split"

In [4]:
from models.cnn_bilstm import train_cnn_bilstm, predict_cnn_bilstm
from evaluation.metrics import compute_metrics_real_scale

In [5]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
pm25_scaler = joblib.load(ARTIFACTS_DIR / "pm25_scaler.pkl")

In [6]:
target_col = "PM2.5"
id_cols = ["Station_No", "date"]

In [ ]:
SEQ_LEN = 48
EPOCHS = 200
BATCH = 64

all_results = []

In [8]:
station_dirs = sorted([p for p in STATION_SPLIT_DIR.glob("station_*") if p.is_dir()],
                      key=lambda x: int(x.name.split("_")[1]))

In [ ]:
for st_dir in station_dirs:
    st = int(st_dir.name.split("_")[1])
    train_path = st_dir / "train.csv"
    val_path   = st_dir / "val.csv"
    test_path  = st_dir / "test.csv"

    train_df = pd.read_csv(train_path); train_df["date"] = pd.to_datetime(train_df["date"])
    val_df   = pd.read_csv(val_path);   val_df["date"]   = pd.to_datetime(val_df["date"])
    test_df  = pd.read_csv(test_path);  test_df["date"]  = pd.to_datetime(test_df["date"])

    feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32)

    X_val = val_df[feature_cols].values.astype(np.float32)
    y_val = val_df[target_col].values.astype(np.float32)

    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32)

    ckpt_path = ARTIFACTS_DIR / f"cnn_bilstm_station{st}_best_seq{SEQ_LEN}_u256.keras"

    # Train
    model, history = train_cnn_bilstm(
        X_train, y_train,
        X_val, y_val,
        seq_len=SEQ_LEN,
        epochs=EPOCHS,
        batch_size=BATCH,
        use_early_stopping=True,
        patience=20,
        learning_rate=3e-4,
        lstm_units=192,
        use_attention=True,
        checkpoint_path=str(ckpt_path)
    )

    # Predict using best checkpoint
    best_model = load_model(ckpt_path)
    y_pred_scaled = predict_cnn_bilstm(best_model, X_test, seq_len=SEQ_LEN)
    y_true_scaled = y_test[SEQ_LEN:]

    metrics = compute_metrics_real_scale(
        y_true_scaled=y_true_scaled,
        y_pred_scaled=y_pred_scaled,
        scaler=pm25_scaler,
        eps=1.0,
        mask_nan=True
    )

    row = {"Station_No": st, "n_test": int(len(y_true_scaled))}
    row.update(metrics)
    all_results.append(row)

    print(f"Done station {st} | Corr={metrics['Correlation']:.3f} RMSE={metrics['RMSE']:.2f} MAE={metrics['MAE']:.2f} MAPE={metrics['MAPE']:.3f}")

Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.1860
Epoch 1: val_loss improved from None to 0.01270, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_best.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 214s 1s/step - loss: 0.0657 - val_loss: 0.0127 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0148
Epoch 2: val_loss improved from 0.01270 to 0.01233, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_best.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 161s 1s/step - loss: 0.0141 - val_loss: 0.0123 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - loss: 0.0117
Epoch 3: val_loss did not improve from 0.01233
126/126 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - loss: 0.0120 - val_loss: 0.0129 - learning_rate: 3.0000e-04
Epoch 4/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.0113
Epoch 4: val_loss improved from 0.01233 to 0.00934, saving mod

In [10]:
results_station_df = pd.DataFrame(all_results).sort_values("Station_No").reset_index(drop=True)
results_station_df

,Station_No,n_test,Correlation,RMSE,MAPE,MAE
0,1,1691,0.872754,41.191441,0.393175,30.371610
1,2,1691,0.864578,24.427860,0.129134,15.843320
2,3,1691,0.852400,35.470944,0.351853,24.578048
3,4,1691,0.876883,40.403712,0.169781,28.341695
4,5,1691,0.943195,21.396153,3.996790,15.906768
5,6,1689,0.899586,24.734059,0.127421,16.493991


In [18]:
print("Station-wise results:")
display(results_station_df)

print("\nMean across stations:")
print(results_station_df[["Correlation","RMSE","MAE","MAPE"]].mean())


Station-wise results:


,Station_No,n_test,Correlation,RMSE,MAPE,MAE
0,1,1691,0.872754,41.191441,0.393175,30.371610
1,2,1691,0.864578,24.427860,0.129134,15.843320
2,3,1691,0.852400,35.470944,0.351853,24.578048
3,4,1691,0.876883,40.403712,0.169781,28.341695
4,5,1691,0.943195,21.396153,3.996790,15.906768
5,6,1689,0.899586,24.734059,0.127421,16.493991



Mean across stations:
Correlation     0.884899
RMSE           31.270695
MAE            21.922572
MAPE            0.861359
dtype: float64


In [12]:
print("MAPE median:", results_station_df["MAPE"].median())


MAPE median: 0.2608167583120065


In [23]:
import importlib
import joblib
from datetime import timedelta
import models.cnn_bilstm as cnn_bilstm

importlib.reload(cnn_bilstm)

<module 'models.cnn_bilstm' from 'C:\\Users\\huyy\\AirPollutionPrediction-CNN-BiLSTM\\models\\cnn_bilstm.py'>

In [24]:
from models.cnn_bilstm import train_cnn_bilstm_v2

In [26]:
print("Station 1 check - X_train shape:", X_train.shape, "y_train shape:", y_train.shape)


Station 1 check - X_train shape: (8112, 32) y_train shape: (8112,)


In [30]:
# ==== OPTIMAL CONFIG FOUND FROM GRID ====
SEQ_LEN = 48
EPOCHS = 200
BATCH = 64
PATIENCE = 20
LSTM_UNITS = 128
LR = 3e-4

all_results = []

for st_dir in station_dirs:
    st = int(st_dir.name.split("_")[1])

    train_df = pd.read_csv(st_dir / "train.csv")
    val_df   = pd.read_csv(st_dir / "val.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")

    feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32)

    X_val = val_df[feature_cols].values.astype(np.float32)
    y_val = val_df[target_col].values.astype(np.float32)

    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32)

    ckpt = ARTIFACTS_DIR / f"cnn_bilstm_station{st}_opt.keras"

    model, _ = train_cnn_bilstm(
        X_train, y_train,
        X_val, y_val,
        seq_len=SEQ_LEN,
        epochs=EPOCHS,
        batch_size=BATCH,
        use_early_stopping=True,
        patience=PATIENCE,
        learning_rate=LR,
        lstm_units=LSTM_UNITS,
        use_attention=True,
        checkpoint_path=str(ckpt)
    )

    best = load_model(ckpt)

    y_pred = predict_cnn_bilstm(best, X_test, seq_len=SEQ_LEN)
    y_true = y_test[SEQ_LEN:]

    m = compute_metrics_real_scale(
        y_true_scaled=y_true,
        y_pred_scaled=y_pred,
        scaler=pm25_scaler,
        eps=1.0,
        mask_nan=True
    )

    row = {"Station_No": st}
    row.update(m)
    all_results.append(row)

    print(f"Station {st} | Corr={m['Correlation']:.3f} RMSE={m['RMSE']:.2f}")

results_opt = pd.DataFrame(all_results)
print("\nMean across stations:")
print(results_opt[["Correlation","RMSE","MAE","MAPE"]].mean())


Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - loss: 0.1227
Epoch 1: val_loss improved from None to 0.01189, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_opt.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 31s 191ms/step - loss: 0.0471 - val_loss: 0.0119 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - loss: 0.0140
Epoch 2: val_loss improved from 0.01189 to 0.01133, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_opt.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 39s 310ms/step - loss: 0.0130 - val_loss: 0.0113 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - loss: 0.0126
Epoch 3: val_loss did not improve from 0.01133
126/126 ━━━━━━━━━━━━━━━━━━━━ 45s 343ms/step - loss: 0.0120 - val_loss: 0.0119 - learning_rate: 3.0000e-04
Epoch 4/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - loss: 0.0114
Epoch 4: val_loss improved from 0.01133 to 0.0

In [31]:
results_opt = pd.DataFrame(all_results)
print("\nMean across stations:")
print(results_opt[["Correlation","RMSE","MAE","MAPE"]].mean())


Mean across stations:
Correlation     0.885042
RMSE           31.178803
MAE            21.682525
MAPE            0.866492
dtype: float64


In [32]:
import numpy as np
from models.cnn_bilstm import train_cnn_bilstm, predict_cnn_bilstm

# =============================
# 1) Convert target to REAL scale
# =============================
# y_train, y_val, y_test hiện đang là scaled
# Ta inverse để về PM2.5 thật

y_train_real = pm25_scaler.inverse_transform(y_train.reshape(-1,1)).ravel()
y_val_real   = pm25_scaler.inverse_transform(y_val.reshape(-1,1)).ravel()
y_test_real  = pm25_scaler.inverse_transform(y_test.reshape(-1,1)).ravel()

# =============================
# 2) Log-transform
# =============================
y_train_log = np.log1p(y_train_real)
y_val_log   = np.log1p(y_val_real)
y_test_log  = np.log1p(y_test_real)

# =============================
# 3) Train model on log target
# =============================
SEQ_LEN = 48
EPOCHS = 200
BATCH = 64

ckpt_log = ARTIFACTS_DIR / "cnn_bilstm_station1_log.keras"

model_log, _ = train_cnn_bilstm(
    X_train, y_train_log,
    X_val, y_val_log,
    seq_len=SEQ_LEN,
    epochs=EPOCHS,
    batch_size=BATCH,
    use_early_stopping=True,
    patience=20,
    learning_rate=3e-4,
    lstm_units=128,
    use_attention=True,
    checkpoint_path=str(ckpt_log)
)

best_log = load_model(ckpt_log)

# =============================
# 4) Predict (log space)
# =============================
y_pred_log = predict_cnn_bilstm(best_log, X_test, seq_len=SEQ_LEN)
y_true_log = y_test_log[SEQ_LEN:]

# =============================
# 5) Convert back to REAL
# =============================
y_pred_real = np.expm1(y_pred_log)
y_true_real = np.expm1(y_true_log)

# =============================
# 6) Compute metrics manually
# =============================
diff = y_pred_real - y_true_real

rmse = np.sqrt(np.mean(diff**2))
mae = np.mean(np.abs(diff))
corr = np.corrcoef(y_pred_real, y_true_real)[0,1]

mask = np.abs(y_true_real) >= 1.0
mape = np.mean(np.abs(diff[mask]) / np.abs(y_true_real[mask]))

print("\nStation 1 - LOG MODEL REAL SCALE:")
print("Correlation:", corr)
print("RMSE:", rmse)
print("MAE:", mae)
print("MAPE:", mape)


Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - loss: 0.5633
Epoch 1: val_loss improved from None to 0.07900, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_log.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 88s 580ms/step - loss: 0.2520 - val_loss: 0.0790 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 891ms/step - loss: 0.1260
Epoch 2: val_loss improved from 0.07900 to 0.04621, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_log.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 121s 960ms/step - loss: 0.1208 - val_loss: 0.0462 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.1142
Epoch 3: val_loss did not improve from 0.04621
126/126 ━━━━━━━━━━━━━━━━━━━━ 160s 1s/step - loss: 0.1113 - val_loss: 0.0763 - learning_rate: 3.0000e-04
Epoch 4/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.1076
Epoch 4: val_loss improved from 0.04621 to 0.03469, s

In [33]:
SEQ_LEN = 48
EPOCHS = 200
BATCH = 64
PATIENCE = 20
LSTM_UNITS = 128
LR = 3e-4

results = []

for st_dir in station_dirs:
    st = int(st_dir.name.split("_")[1])

    train_df = pd.read_csv(st_dir / "train.csv")
    val_df   = pd.read_csv(st_dir / "val.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")

    feature_cols = [c for c in train_df.columns if c not in id_cols + [target_col]]

    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32)

    X_val = val_df[feature_cols].values.astype(np.float32)
    y_val = val_df[target_col].values.astype(np.float32)

    X_test = test_df[feature_cols].values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32)

    # ===== y: scaled -> real -> log1p =====
    y_train_real = pm25_scaler.inverse_transform(y_train.reshape(-1,1)).ravel()
    y_val_real   = pm25_scaler.inverse_transform(y_val.reshape(-1,1)).ravel()
    y_test_real  = pm25_scaler.inverse_transform(y_test.reshape(-1,1)).ravel()

    y_train_log = np.log1p(np.clip(y_train_real, 0, None))
    y_val_log   = np.log1p(np.clip(y_val_real, 0, None))
    y_test_log  = np.log1p(np.clip(y_test_real, 0, None))

    ckpt = ARTIFACTS_DIR / f"cnn_bilstm_station{st}_LOG_seq{SEQ_LEN}_u{LSTM_UNITS}.keras"

    # ===== train in log space =====
    model, _ = train_cnn_bilstm(
        X_train, y_train_log,
        X_val, y_val_log,
        seq_len=SEQ_LEN,
        epochs=EPOCHS,
        batch_size=BATCH,
        use_early_stopping=True,
        patience=PATIENCE,
        learning_rate=LR,
        lstm_units=LSTM_UNITS,
        use_attention=True,
        checkpoint_path=str(ckpt)
    )

    best = load_model(ckpt)

    # ===== predict log -> real =====
    y_pred_log = predict_cnn_bilstm(best, X_test, seq_len=SEQ_LEN)
    y_true_log = y_test_log[SEQ_LEN:]

    y_pred_real = np.expm1(y_pred_log)
    y_true_real = np.expm1(y_true_log)

    # clip negative predictions (safety)
    y_pred_real = np.clip(y_pred_real, 0, None)

    # ===== metrics real scale =====
    diff = y_pred_real - y_true_real
    rmse = float(np.sqrt(np.mean(diff**2)))
    mae  = float(np.mean(np.abs(diff)))

    if np.std(y_true_real) < 1e-6 or np.std(y_pred_real) < 1e-6:
        corr = float("nan")
    else:
        corr = float(np.corrcoef(y_pred_real, y_true_real)[0,1])

    mask = np.abs(y_true_real) >= 1.0
    mape = float(np.mean(np.abs(diff[mask]) / np.abs(y_true_real[mask]))) if np.any(mask) else float("nan")

    results.append({
        "Station_No": st,
        "n_test": int(len(y_true_real)),
        "Correlation": corr,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    })

    print(f"Station {st} | Corr={corr:.3f} RMSE={rmse:.2f} MAE={mae:.2f} MAPE={mape:.3f}")

results_log_df = pd.DataFrame(results).sort_values("Station_No").reset_index(drop=True)
print("\nMean across stations (LOG-target):")
print(results_log_df[["Correlation","RMSE","MAE","MAPE"]].mean())

results_log_df

Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - loss: 0.5034
Epoch 1: val_loss improved from None to 0.13071, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_LOG_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 114s 717ms/step - loss: 0.2551 - val_loss: 0.1307 - learning_rate: 3.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 682ms/step - loss: 0.1454
Epoch 2: val_loss improved from 0.13071 to 0.11060, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_station1_LOG_seq48_u128.keras
126/126 ━━━━━━━━━━━━━━━━━━━━ 92s 734ms/step - loss: 0.1429 - val_loss: 0.1106 - learning_rate: 3.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 627ms/step - loss: 0.1274
Epoch 3: val_loss did not improve from 0.11060
126/126 ━━━━━━━━━━━━━━━━━━━━ 85s 679ms/step - loss: 0.1260 - val_loss: 0.1434 - learning_rate: 3.0000e-04
Epoch 4/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - loss: 0.1199
Epoch 4: val_loss impro

,Station_No,n_test,Correlation,RMSE,MAE,MAPE
0,1,1691,0.850262,45.962128,31.457357,0.318530
1,2,1691,0.866036,24.249121,15.471744,0.124699
2,3,1691,0.874539,32.602165,21.116827,0.246614
3,4,1691,0.880597,40.039757,26.946671,0.145984
4,5,1691,0.945971,20.337324,14.977503,0.132320
5,6,1689,0.900549,24.954287,16.541933,0.124064


In [34]:
print(train_df.shape, val_df.shape, test_df.shape)
print(train_df["date"].min(), train_df["date"].max())


(8098, 35) (1735, 35) (1737, 35)
2021-02-24 16:00:00 2022-01-28 01:00:00
